# Exploratory Data Analysis (EDA) Report
## Project: House Price Prediction using Linear Regression

This notebook contains a professional, production-grade exploratory data analysis of the House Prices dataset. The goal is to examine the dataset structure, detect anomalies/missing data, analyze the target variable (`SalePrice`), and uncover key relationships between features and the target variable to guide our preprocessing and model-building pipelines.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
from pathlib import Path

# Ensure project root in path
sys.path.append(str(Path(os.getcwd()).resolve().parent))
import config

# Configure plotting aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

### 1. Dataset Overview & Ingestion
We load the training dataset from the configured path and inspect its shape and column structure.

In [ ]:
# Load dataset
df_raw = pd.read_csv(config.TRAIN_PATH)
print(f"Raw dataset shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
df_raw.head()

### 2. Feature Extraction & Standardisation
We apply our data loading mappings to align with the core model features: `GrLivArea` (Square Footage), `BedroomAbvGr` (Bedrooms), and `FullBath` + `HalfBath` (Bathrooms).

In [ ]:
from src.data_loader import DataLoader
loader = DataLoader()
df_std = loader.load_data(config.TRAIN_PATH, is_training=True)
df_std.head()

### 3. Data Types & Summary Statistics
We calculate core descriptive statistics for each standardized column to understand the ranges, distributions, and scaling needs.

In [ ]:
print("--- Data Types and Missing Info ---")
print(df_std.info())

print("\n--- Summary Statistics ---")
df_std.describe()

**Insights:**
- **Square Footage (`sqft`)**: Typically ranges from 600 to 5000+ sqft. We need to check if there are extreme outliers.
- **Bedrooms (`bedrooms`)**: Counts range between 1 and 6. Standard house size is around 3 bedrooms.
- **Bathrooms (`bathrooms`)**: Combines full and half bathrooms. Ranges from 1 to 5. 
- **Price (`price`)**: The target variable. Prices have a large range, which is common in housing markets. Scaling the inputs will be helpful to ensure weights are stable.

### 4. Missing Values & Duplicate Rows
We check for missing data and duplicates to ensure data quality.

In [ ]:
missing_values = df_std.isnull().sum()
duplicates_count = df_std.duplicated().sum()

print("Missing values per column:")
print(missing_values)
print(f"\nDuplicate rows found: {duplicates_count}")

### 5. Target Variable Analysis (`price`)
We plot the distribution of the target variable `price` to inspect skewness and normality. Linear regression models assume residuals are normally distributed; skewed target variables can sometimes affect regression fit.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# Histogram & KDE
sns.histplot(df_std["price"], kde=True, color="#4F46E5", ax=ax[0])
ax[0].set_title("Distribution of House Prices (Histogram + KDE)", fontweight="bold")
ax[0].set_xlabel("Price ($)")
ax[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"${x:,.0f}"))

# Box Plot for Outliers
sns.boxplot(x=df_std["price"], color="#10B981", ax=ax[1])
ax[1].set_title("Boxplot of House Prices (Outlier Detection)", fontweight="bold")
ax[1].set_xlabel("Price ($)")
ax[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"${x:,.0f}"))

plt.tight_layout()
plt.show()

**Insights:**
- The target distribution is slightly right-skewed, which is typical for housing prices (a few very expensive luxury homes pull the mean up).
- The boxplot indicates several outliers on the upper end (luxury properties). We should evaluate whether to remove extreme outliers in training to prevent them from overly pulling the regression line.

### 6. Correlation Analysis
We construct a correlation matrix and visualize it as a heatmap to inspect collinearity and identify features strongly linked to house prices.

In [ ]:
corr_matrix = df_std.corr()
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5, vmin=-1, vmax=1)
plt.title("Feature Correlation Heatmap Matrix", fontsize=13, fontweight="bold", pad=15)
plt.show()

**Insights:**
- **Square Footage (`sqft`)** shares the strongest positive correlation with `price` (expected, as house size is a primary driver of cost).
- **Bathrooms (`bathrooms`)** and **Bedrooms (`bedrooms`)** also show positive correlations with `price`.
- The features themselves have moderate positive correlation with each other (collinearity). Standard scaling will help the optimization algorithm.

### 7. Feature Relationships
We plot joint relationships between each feature and price to visually confirm linearity.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))

# 1. Sqft vs Price
sns.regplot(data=df_std, x="sqft", y="price", scatter_kws={"alpha": 0.5, "color": "#4F46E5"}, line_kws={"color": "#EF4444"}, ax=ax[0])
ax[0].set_title("Square Footage vs. Price", fontweight="bold")
ax[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"${x:,.0f}"))

# 2. Bedrooms vs Price
sns.boxplot(data=df_std, x="bedrooms", y="price", palette="Blues", ax=ax[1])
ax[1].set_title("Bedrooms vs. Price", fontweight="bold")
ax[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"${x:,.0f}"))

# 3. Bathrooms vs Price
sns.boxplot(data=df_std, x="bathrooms", y="price", palette="Greens", ax=ax[2])
ax[2].set_title("Bathrooms vs. Price", fontweight="bold")
ax[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"${x:,.0f}"))

plt.tight_layout()
plt.show()

**Insights:**
- **Sqft**: Shows a clear, strong linear relationship with price. Scatter points are evenly spread, though dispersion increases for larger houses.
- **Bedrooms & Bathrooms**: Average house prices increase step-wise as bedroom and bathroom counts rise, confirming positive coefficients for our regression model.

### 8. Key EDA Summary & Action Items for Preprocessing
1. **Linear Relationship**: Confirmed that all numerical features display linear tendencies with respect to price, validating our selection of a Linear Regression model.
2. **Outlier Filtration**: Extreme values in `sqft` (e.g., houses > 4000 sqft) and `price` can exert disproportionate leverage on a standard OLS regression. We will apply IQR cleaning during training to stabilize parameters.
3. **Missing Value Management**: Imputing using standard strategies (median for numerical) is safe since missing values are rare.
4. **Scaling**: Standard scaling of independent numerical features will ensure coefficients are easy to compare and interpret directly.